# NBA Playoff Predictor Model

- Customizable neural network via GUI
- Model outputs probability of the home team winning the matchup
- Trained on team's regular season statistics

## Table of Contents
- [1. Select Season and Create Custom NN with GUI](#1-select-season-and-create-custom-nn-with-gui)
- [2. Prepare training data](#2-prepare-training-data)
- [3. Baseline: Logistic Regression](#3-baseline-logistic-regression)
- [4. Custom Neural Network](#4-custom-neural-network)
- [5. Train GUI-defined model](#5-train-gui-defined-model)
- [6. Predicting the Playoffs](#6-predicting-the-playoffs)
- [7. Playoff Simulation](#7-playoff-simulation)
  - [7.1 Round 1](#71-round-1)
  - [7.2 Conference Semifinals](#72-conference-semifinals)
  - [7.3 Conference Finals](#73-conference-finals)
  - [7.4 NBA Finals](#74-nba-finals)

In [35]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import utils, models, layers, metrics
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from gui import create_model_gui
from datetime import date
from seasons_matchups import NBA_SEASONS, PLAYOFF_MATCHUPS
from game_data_ingest import *

## 1. Select Season and Create Custom NN with GUI

In [4]:
model, season = create_model_gui()
print(f"\nSeason To Simulate: {season}")

c:\Users\nyc8p\anaconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,353 (17.00 KB)

 Trainable params: 4,353 (17.00 KB)

 Non-trainable params: 0 (0.00 B)


Season To Simulate: 2000-2001


In [5]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,353 (17.00 KB)

 Trainable params: 4,353 (17.00 KB)

 Non-trainable params: 0 (0.00 B)

## 2. Prepare training data
Query the database to get our training data. Each row in the dataframe corresponds to the average statistics of the home and away team before their game, as well as the outcome of the game they played.

In [9]:
# add selected season to database
season_start = NBA_SEASONS[season]['start']
season_end = NBA_SEASONS[season]['end']
some_dates = date_range(season_start, season_end)
put_dates_in_db(some_dates)

with sqlite3.connect("games.db") as conn:
    total_games, = conn.execute("SELECT COUNT(*) FROM games").fetchone()
    df = generate_training_data_for_season(conn, season_start, season_end)

print(f"Total count of games in the database: {total_games}")
print("---- Training Data ----")
print(df)

5 dates have to be scraped
Starting crawl
Finished crawl
Total count of games in the database: 1574
---- Training Data ----
      hometeam_fieldGoalsMade  hometeam_fieldGoalsAttempted  \
0                   35.524590                     78.786885   
1                   37.642857                     82.875000   
2                   38.338983                     81.864407   
3                   36.482759                     77.810345   
4                   34.983607                     79.229508   
...                       ...                           ...   
1168                34.844156                     80.909091   
1169                33.227848                     78.455696   
1170                33.833333                     79.166667   
1171                34.233766                     78.792208   
1172                35.571429                     83.597403   

      hometeam_threePointersMade  hometeam_threePointersAttempted  \
0                       6.098361                  

In [23]:
df.head()

,hometeam_fieldGoalsMade,hometeam_fieldGoalsAttempted,hometeam_threePointersMade,hometeam_threePointersAttempted,hometeam_freeThrowsMade,hometeam_freeThrowsAttempted,hometeam_reboundsOffensive,hometeam_reboundsDefensive,hometeam_assists,hometeam_steals,...,awayteam_freeThrowsMade,awayteam_freeThrowsAttempted,awayteam_reboundsOffensive,awayteam_reboundsDefensive,awayteam_assists,awayteam_steals,awayteam_blocks,awayteam_foulsPersonal,awayteam_points,winner_is_home_team
0,35.524590,78.786885,6.098361,17.016393,19.196721,25.311475,11.327869,30.426230,19.409836,7.360656,...,17.126984,24.936508,12.222222,31.142857,18.873016,6.063492,6.301587,21.952381,91.555556,1
1,37.642857,82.875000,6.910714,17.750000,18.089286,23.178571,11.964286,30.267857,22.125000,8.178571,...,19.103448,25.189655,11.293103,30.431034,19.379310,7.310345,4.689655,20.913793,96.034483,0
2,38.338983,81.864407,5.305085,15.661017,19.033898,28.305085,13.271186,31.491525,22.881356,6.508475,...,18.836066,25.081967,13.868852,30.983607,24.032787,7.098361,5.754098,21.754098,97.622951,1
3,36.482759,77.810345,4.465517,12.586207,17.965517,23.310345,11.620690,30.362069,23.810345,8.551724,...,17.150000,25.083333,12.383333,31.316667,19.083333,6.033333,6.350000,22.266667,91.666667,1
4,34.983607,79.229508,4.344262,13.114754,17.180328,25.098361,12.262295,31.262295,19.049180,6.016393,...,18.767857,24.517857,12.232143,31.964286,22.875000,9.785714,5.678571,19.535714,101.107143,1


In [20]:
df['winner_is_home_team'].value_counts()

winner_is_home_team
1    702
0    471
Name: count, dtype: int64

We split our training data into three sets with a 75/15/10 ratio for train/validation/test. Since home teams win ~60% of the time (class imbalance), we'll also compute class weights to ensure the model learns to predict both home and away wins effectively.


**Split breakdown:**
- Train: 75%
- Validation: 15%
- Test: 10%

**Features:** 26 total (13 stats * 2 teams with `hometeam_` and `awayteam_` prefixes)

**Target:** `winner_is_home_team` (1 = home win, 0 = away win)



In [109]:
train_ratio = 0.75
validation_ratio = 0.15
test_ratio = 0.10

X = df.drop('winner_is_home_team', axis=1).values
y = df['winner_is_home_team'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, shuffle=True, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio/(test_ratio + validation_ratio), shuffle=True, stratify=y_temp)

# class weights to account for class imbalance
# class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
# class_weight_dict = dict(enumerate(class_weights))
# NOTE: using the above class weight of {0: ~1.24, 1: ~0.85} caused model to heavily favor predicting 0
# NOTE: therefore, use manually set weights that aren't as extreme
class_weight_dict = {0: 1.1, 1: 0.9}
print(f"Class weights: {class_weight_dict}")

# reshape y's 
y_train = y_train.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

Class weights: {0: 1.1, 1: 0.9}


## 3. Baseline: Logistic Regression
Before we create our model, we test the predictive performance of a simple Logistic Regression model first.

In [91]:
model_logistic_regression = models.Sequential([
    # Logistic Regression
    layers.InputLayer(input_shape=(26,)),
    layers.Dense(units=1, activation='sigmoid')
])

model_logistic_regression.summary()

c:\Users\nyc8p\anaconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 1)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27 (108.00 B)

 Trainable params: 27 (108.00 B)

 Non-trainable params: 0 (0.00 B)

In [94]:
model_logistic_regression.compile(optimizer='adam', 
              loss=tf.keras.losses.binary_crossentropy, 
              metrics=['accuracy', 
                       metrics.Precision(), 
                       metrics.Recall(),
                       metrics.F1Score(threshold=0.5),
                       metrics.AUC()])
earlystopping = EarlyStopping(patience=5)
history_logistic_regression = model_logistic_regression.fit(x=X_train,y=y_train,
                     epochs=20,
                     validation_data=(X_val,y_val),
                     class_weight=class_weight_dict,
                     callbacks=[earlystopping])

Epoch 1/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5137 - auc_15: 0.5133 - f1_score: 0.5421 - loss: 1.1091 - precision_32: 0.5796 - recall_32: 0.5149 - val_accuracy: 0.5170 - val_auc_15: 0.4894 - val_f1_score: 0.5894 - val_loss: 1.1583 - val_precision_32: 0.5980 - val_recall_32: 0.5810
Epoch 2/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5638 - auc_15: 0.5121 - f1_score: 0.6476 - loss: 1.0912 - precision_32: 0.6356 - recall_32: 0.6612 - val_accuracy: 0.5284 - val_auc_15: 0.4903 - val_f1_score: 0.5744 - val_loss: 1.1676 - val_precision_32: 0.6222 - val_recall_32: 0.5333
Epoch 3/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4784 - auc_15: 0.4567 - f1_score: 0.5501 - loss: 1.1647 - precision_32: 0.5866 - recall_32: 0.5187 - val_accuracy: 0.5170 - val_auc_15: 0.4904 - val_f1_score: 0.5933 - val_loss: 1.1401 - val_precision_32: 0.5962 - val_recall_32: 0.5905
Epoch 4/20
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5001 - auc_15: 0.4679 - f1_score: 0

In [95]:
results_logistic = model_logistic_regression.evaluate(x=X_test,y=y_test)
print("\n--- Logistic Regression Results ---")
print(f"Loss: {results_logistic[0]:.4f}")
print(f"Accuracy: {results_logistic[1]:.4f}")
print(f"Precision: {results_logistic[2]:.4f}")
print(f"Recall: {results_logistic[3]:.4f}")
print(f"F1 Score: {results_logistic[4]:.4f}")
print(f"AUC: {results_logistic[5]:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5683 - auc_15: 0.5029 - f1_score: 0.6414 - loss: 0.9253 - precision_32: 0.6554 - recall_32: 0.6292

--- Logistic Regression Results ---
Loss: 0.9263
Accuracy: 0.5847
Precision: 0.6528
Recall: 0.6620
F1 Score: 0.6573
AUC: 0.5265


## 4. Custom Neural Network
Now we try our own Neural Network model.

In [96]:
model1 = models.Sequential([
    # NN
    layers.InputLayer(input_shape=(26,)),
    layers.Dense(units=128, use_bias=True), # Linear layer
    layers.Dense(units=64, use_bias=True, activation='relu'), # Non-Linear layer
    layers.Dense(units=64, activation='relu'), # relu
    layers.Dense(units=1, activation='sigmoid') # output layer: probability of home team win
])

model1.summary()

c:\Users\nyc8p\anaconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_21 (Dense)                │ (None, 128)            │         3,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,937 (62.25 KB)

 Trainable params: 15,937 (62.25 KB)

 Non-trainable params: 0 (0.00 B)

In [100]:
model1.compile(optimizer='adam',
              loss=tf.keras.losses.binary_crossentropy,
              metrics=['accuracy', 
                       metrics.Precision(), 
                       metrics.Recall(),
                       metrics.F1Score(threshold=0.5),
                       metrics.AUC()])

history_1 = model1.fit(x=X_train,y=y_train,
                       epochs=30,
                       validation_data=(X_val,y_val),
                       class_weight=class_weight_dict,
                       callbacks=[earlystopping]
                     )

Epoch 1/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.6105 - auc_18: 0.6324 - f1_score: 0.6742 - loss: 0.7780 - precision_35: 0.6930 - recall_35: 0.6617 - val_accuracy: 0.5909 - val_auc_18: 0.6743 - val_f1_score: 0.5714 - val_loss: 0.6718 - val_precision_35: 0.7619 - val_recall_35: 0.4571
Epoch 2/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6646 - auc_18: 0.6814 - f1_score: 0.7289 - loss: 0.6333 - precision_35: 0.7257 - recall_35: 0.7378 - val_accuracy: 0.4432 - val_auc_18: 0.6700 - val_f1_score: 0.1250 - val_loss: 0.8520 - val_precision_35: 1.0000 - val_recall_35: 0.0667
Epoch 3/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5468 - auc_18: 0.6013 - f1_score: 0.5871 - loss: 0.7115 - precision_35: 0.6224 - recall_35: 0.5688 - val_accuracy: 0.5966 - val_auc_18: 0.6494 - val_f1_score: 0.7455 - val_loss: 0.6801 - val_precision_35: 0.5977 - val_recall_35: 0.9905
Epoch 4/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5290 - auc_18: 0.5596 - f1_score: 0

In [101]:
results_custom_model = model1.evaluate(x=X_test,y=y_test)
print("\n--- Custom Model Results ---")
print(f"Loss: {results_custom_model[0]:.4f}")
print(f"Accuracy: {results_custom_model[1]:.4f}")
print(f"Precision: {results_custom_model[2]:.4f}")
print(f"Recall: {results_custom_model[3]:.4f}")
print(f"F1 Score: {results_custom_model[4]:.4f}")
print(f"AUC: {results_custom_model[5]:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6073 - auc_18: 0.6179 - f1_score: 0.7019 - loss: 0.6610 - precision_35: 0.6566 - recall_35: 0.7541

--- Custom Model Results ---
Loss: 0.6616
Accuracy: 0.6017
Precision: 0.6500
Recall: 0.7324
F1 Score: 0.6887
AUC: 0.6220


## 5. Train GUI-defined model
We also train the model configured via the GUI.

In [102]:
model.summary() # 3 Hidden Layers, Relu, Linear, Relu

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,061 (51.02 KB)

 Trainable params: 4,353 (17.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,708 (34.02 KB)

In [103]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.binary_crossentropy,
              metrics=['accuracy', metrics.Precision(), metrics.Recall()]) # metrics to track

stop_here_please = EarlyStopping(patience=7)

history_1 = model.fit(x=X_train,y=y_train,
                     epochs=30,
                     validation_data=(X_val,y_val),
                     class_weight=class_weight_dict,
                     callbacks=[stop_here_please]
                     )

Epoch 1/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5835 - loss: 0.6689 - precision_36: 0.6606 - recall_36: 0.6795 - val_accuracy: 0.6136 - val_loss: 0.6644 - val_precision_36: 0.6108 - val_recall_36: 0.9714
Epoch 2/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6498 - loss: 0.6400 - precision_36: 0.6776 - recall_36: 0.7896 - val_accuracy: 0.6136 - val_loss: 0.6302 - val_precision_36: 0.6412 - val_recall_36: 0.8000
Epoch 3/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6289 - loss: 0.6236 - precision_36: 0.6647 - recall_36: 0.7533 - val_accuracy: 0.6307 - val_loss: 0.6339 - val_precision_36: 0.6818 - val_recall_36: 0.7143
Epoch 4/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6508 - loss: 0.6251 - precision_36: 0.6926 - recall_36: 0.7672 - val_accuracy: 0.6136 - val_loss: 0.6334 - val_precision_36: 0.6581 - val_recall_36: 0.7333
Epoch 5/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6490 - loss: 0.6276 - precision_36: 0.7027 - recall_36:

In [104]:
results = model.evaluate(x=X_test, y=y_test)
print(f"Loss: {results[0]:.4f}")
print(f"Accuracy: {results[1]:.4f}")
print(f"Precision: {results[2]:.4f}")
print(f"Recall: {results[3]:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6123 - loss: 0.6742 - precision_36: 0.7298 - recall_36: 0.5823
Loss: 0.6690
Accuracy: 0.5932
Precision: 0.7091
Recall: 0.5493


## 6. Predicting the Playoffs

Now that we have the model and the season, we now can test the model on each matchup in the postseason and see how it predicts the NBA playoffs!

In [106]:
def get_matchup_df(conn, team1, team2):
    '''
    Creates concatenated dataframe of the regular season averaged data of two specified teams.
    '''
    season_start = NBA_SEASONS[season]['start']
    season_end = NBA_SEASONS[season]['end']
    df1 = get_average_stats_over_period(conn, team1, season_start, season_end)
    df2 = get_average_stats_over_period(conn, team2, season_start, season_end)
    df1 = df1.drop('gamesCount', axis = 1)
    df2 = df2.drop('gamesCount', axis = 1)
    df1 = df1.add_prefix("hometeam_")
    df2 = df2.add_prefix("awayteam_")
    resultdf = pd.concat([df1,df2], axis = 1)
    return resultdf

def predict_match(conn, model, home_team, away_team):
    '''
    Predicts winner of a matchup. 
    Probability > 0.5 => home team wins.
    '''
    # home_team is first; >0.5 means first team wins
    df = get_matchup_df(conn, home_team, away_team)
    prob = float(model.predict(df, verbose=0)[0][0])
    winner = home_team if prob > 0.5 else away_team
    return winner, prob

def simulate_round(conn, model, teams, round_name):
    # teams is a flat list in pair order: [t1, t2, t3, t4, ...]
    logs = []
    winners = []
    assert len(teams) % 2 == 0, f"{round_name}: team list must be even, got {len(teams)}"
    for i in range(0, len(teams), 2):
        home, away = teams[i], teams[i+1]
        winner, prob = predict_match(conn, model, home, away)
        winners.append(winner)
        logs.append({
            "round": round_name,
            "home": home,
            "away": away,
            "p_home_wins": prob,
            "winner": winner
        })
    return winners, logs

def simulate_conference(conn, model, teams, conference_name):
    all_logs = []
    current = teams[:]
    rnd = 1
    while len(current) > 1:
        round_name = f"{conference_name} R{rnd} ({len(current)} teams)"
        current, logs = simulate_round(conn, model, current, round_name)
        all_logs.extend(logs)
        rnd += 1
    champ = current[0]
    return champ, all_logs

def simulate_playoffs(conn, model, season):
    east = PLAYOFF_MATCHUPS[season]["East"]
    west = PLAYOFF_MATCHUPS[season]["West"]

    east_champ, east_logs = simulate_conference(conn, model, east, "East")
    west_champ, west_logs = simulate_conference(conn, model, west, "West")

    # NBA Finals: East champ is first (home) by convention here
    finals_winner, p_home = predict_match(conn, model, east_champ, west_champ)
    finals_log = [{
        "round": "NBA Finals",
        "home": east_champ,
        "away": west_champ,
        "p_home_wins": p_home,
        "winner": finals_winner
    }]

    return {
        "east_champ": east_champ,
        "west_champ": west_champ,
        "finals_winner": finals_winner,
        "logs": pd.DataFrame(east_logs + west_logs + finals_log)
    }

In [105]:
sim = simulate_playoffs(conn, model, season)

print(f"East Champion: {sim['east_champ']}")
print(f"West Champion: {sim['west_champ']}")
print(f"NBA Champion:  {sim['finals_winner']}")
display(sim["logs"])

East Champion: bucks
West Champion: kings
NBA Champion:  kings


,round,home,away,p_home_wins,winner
0,East R1 (8 teams),sixers,pacers,0.360761,pacers
1,East R1 (8 teams),hornets,heat,0.420963,heat
2,East R1 (8 teams),raptors,knicks,0.689516,raptors
3,East R1 (8 teams),bucks,magic,0.632419,bucks
4,East R2 (4 teams),pacers,heat,0.506235,pacers
5,East R2 (4 teams),raptors,bucks,0.406052,bucks
6,East R3 (2 teams),pacers,bucks,0.342077,bucks
7,West R1 (8 teams),spurs,timberwolves,0.648563,spurs
8,West R1 (8 teams),lakers,blazers,0.520906,lakers
9,West R1 (8 teams),kings,suns,0.717288,kings


# OLD MANUAL SIMULATION BELOW

In [107]:
lakers_vs_blazers = get_matchup_df(conn, "lakers", "blazers")
mavs_vs_jazz = get_matchup_df(conn, "mavericks", "jazz")
kings_vs_suns = get_matchup_df(conn, "kings", "suns")
spurs_vs_wolves = get_matchup_df(conn, "spurs", "timberwolves")
hornets_vs_heat = get_matchup_df(conn, "hornets", "heat")
bucks_vs_magic = get_matchup_df(conn, "bucks", "magic")
sixers_vs_pacers = get_matchup_df(conn, "sixers", "pacers")
raptors_vs_knicks = get_matchup_df(conn, "raptors", "knicks")

## 7. Playoff Simulation

## Round 1:

In [81]:
model1.predict(lakers_vs_blazers)

1/1 [==============================] - 0s 51ms/step


array([[0.6373418]], dtype=float32)

In [82]:
model.predict(lakers_vs_blazers) #Actual result: Lakers W 3-0

1/1 [==============================] - 0s 70ms/step


array([[0.66255695]], dtype=float32)

In [83]:
model.predict(mavs_vs_jazz) # Actual result: Mavs W 3-2

1/1 [==============================] - 0s 22ms/step


array([[0.6447607]], dtype=float32)

In [84]:
model.predict(kings_vs_suns) # Actual result: Kings W 3-1

1/1 [==============================] - 0s 21ms/step


array([[0.8448413]], dtype=float32)

In [85]:
model.predict(spurs_vs_wolves) # Actual Result: Spurs W 3-1

1/1 [==============================] - 0s 19ms/step


array([[0.76108074]], dtype=float32)

In [86]:
model.predict(hornets_vs_heat) # Actual Result: Hornets W 3-0

1/1 [==============================] - 0s 23ms/step


array([[0.6066924]], dtype=float32)

In [87]:
model.predict(bucks_vs_magic) #Actual Result: Bucks W 3-1

1/1 [==============================] - 0s 20ms/step


array([[0.6545932]], dtype=float32)

In [88]:
model.predict(sixers_vs_pacers) #Actual Result: Sixers W 3-1

1/1 [==============================] - 0s 18ms/step


array([[0.58582693]], dtype=float32)

In [89]:
model.predict(raptors_vs_knicks) #Actual Result: Raptors W 3-2

1/1 [==============================] - 0s 21ms/step


array([[0.83017826]], dtype=float32)

## Conference Semifinals

In [95]:
lakers_vs_kings = get_matchup_df(conn, "lakers", "kings")
spurs_vs_mavs = get_matchup_df(conn, "spurs", "mavericks")
sixers_vs_raptors = get_matchup_df(conn, "sixers", "raptors")
bucks_vs_hornets = get_matchup_df(conn, "bucks", "hornets")

In [91]:
model.predict(lakers_vs_kings) # Actual Result: Lakers W 4-0

1/1 [==============================] - 0s 51ms/step


array([[0.57875705]], dtype=float32)

In [93]:
model.predict(spurs_vs_mavs) # Actual Result: Spurs W 4-1

1/1 [==============================] - 0s 18ms/step


array([[0.6902155]], dtype=float32)

In [96]:
model.predict(sixers_vs_raptors) #Actual Result: Sixers W 4-3 

1/1 [==============================] - 0s 19ms/step


array([[0.5131549]], dtype=float32)

In [97]:
model.predict(bucks_vs_hornets) # Actual Result: Bucks W 4-3

1/1 [==============================] - 0s 27ms/step


array([[0.7862208]], dtype=float32)

## Conference Finals

In [98]:
lakers_vs_spurs = get_matchup_df(conn, "lakers", "spurs")
sixers_vs_bucks = get_matchup_df(conn, "sixers", "bucks")

In [99]:
model.predict(lakers_vs_spurs) # Actual Result: Lakers W 4-0

1/1 [==============================] - 0s 18ms/step


array([[0.5604064]], dtype=float32)

In [100]:
model.predict(sixers_vs_bucks) # Actual Result: Sixers W 4-3 First Incorrect Prediction :(

1/1 [==============================] - 0s 23ms/step


array([[0.49856293]], dtype=float32)

## NBA Finals

In [101]:
model.predict(get_matchup_df(conn, "lakers", "bucks")) # Actual Result, Lakers beat sixers 4-1

1/1 [==============================] - 0s 17ms/step


array([[0.6102966]], dtype=float32)

In [102]:
conn.close()